## Pastikan Sudah Install Dependencies

In [7]:
%pip install langchain langchain-groq sqlalchemy python-dotenv -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Setup API Key(Groq)

In [8]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.getenv("GROQ_API_KEY"):
    print("✅ API key loaded dari .env")
else:
    # Fallback: input manual
    os.environ["GROQ_API_KEY"] = input("Masukkan GROQ_API_KEY kamu: ")
    print("✅ API key berhasil diset")

✅ API key loaded dari .env


## Setup Path & Import

In [9]:
import sys
sys.path.append("..") # agar bisa import dari root project

from src.ai.agent import run_agent
print("✅ Agent siap digunakan")

✅ Agent siap digunakan


## Inisialisasi Database

In [10]:
import sys
sys.path.append("..")

from database.connection import engine
from database.models import Base

Base.metadata.create_all(bind=engine)
print("✅ Tabel database berhasil dibuat")

# Opsional: isi data dummy supaya ada yang bisa ditanya
# %run ../data/dummy/generate_data.py

✅ Tabel database berhasil dibuat


## Simulasi Chatbot

In [ ]:
import json
from datetime import datetime

chat_history = []

print("=" * 50)
print("🤖 SAKU Assistant - Demo Chatbot")
print("=" * 50)
print("Perintah khusus:")
print("  'exit'    → keluar dari chat")
print("  'reset'   → hapus riwayat percakapan")
print("  'history' → tampilkan riwayat chat")
print("  'export'  → simpan riwayat ke JSON")
print("=" * 50)
print("SAKU: Halo! Aku SAKU Assistant 👋 Ada yang bisa aku bantu?\n")

while True:
    try:
        user_input = input("Kamu: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nSampai jumpa! 👋")
        break

    if not user_input:
        continue

    # ── Special Commands ──────────────────────────
    if user_input.lower() == "exit":
        print("SAKU: Sampai jumpa! Semoga usahamu lancar 👋")
        break

    elif user_input.lower() == "reset":
        chat_history = []
        print("SAKU: ✅ Riwayat chat berhasil direset. Mulai percakapan baru!\n")
        continue

    elif user_input.lower() == "history":
        if not chat_history:
            print("SAKU: Belum ada riwayat percakapan.\n")
        else:
            print("\n── Riwayat Chat ──")
            for msg in chat_history:
                label = "Kamu" if msg["role"] == "user" else "SAKU"
                print(f"{label}: {msg['content']}")
            print("──────────────────\n")
        continue

    elif user_input.lower() == "export":
        filename = f"saku_chat_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        data = {
            "diekspor_pada": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "jumlah_pesan": len(chat_history),
            "percakapan": chat_history
        }
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"SAKU: ✅ Riwayat chat disimpan ke '{filename}'\n")
        continue

    # ── Proses ke Agent ───────────────────────────
    # Catat pesan user dulu sebelum proses
    chat_history.append({"role": "user", "content": user_input})
    print(f"User: {user_input}\n")

    try:
        response = run_agent(
            user_input=user_input,
            chat_history=chat_history[:-1]  # kirim history tanpa pesan terakhir
        )
    except Exception as e:
        response = f"⚠️ Terjadi error: {str(e)}. Coba lagi ya."

    # Baru catat respons assistant
    chat_history.append({"role": "assistant", "content": response})

    print(f"SAKU: {response}\n")

🤖 SAKU Assistant - Demo Chatbot
Perintah khusus:
  'exit'    → keluar dari chat
  'reset'   → hapus riwayat percakapan
  'history' → tampilkan riwayat chat
  'export'  → simpan riwayat ke JSON
SAKU: Halo! Aku SAKU Assistant 👋 Ada yang bisa aku bantu?

User: Halo



> Entering new AgentExecutor chain...
Hai! 👋 Ada yang bisa saya bantu hari ini? Mau catat pengeluaran, pemasukan, hutang, atau ada pertanyaan tentang keuangan warung? 😊

> Finished chain.
SAKU: Hai! 👋 Ada yang bisa saya bantu hari ini? Mau catat pengeluaran, pemasukan, hutang, atau ada pertanyaan tentang keuangan warung? 😊

User: Bagaimana keuangan bulan ini?



> Entering new AgentExecutor chain...

Invoking: `tool_ringkasan_keuangan` with `{'periode': 'bulan_ini'}`


📊 Ringkasan Keuangan (bulan ini):
                💰 Total Penjualan    : Rp0
                ➕ Pemasukan Lain     : Rp0
                💸 Total Pengeluaran  : Rp0
                📈 Laba Bersih        : Rp0📊 Ringkasan Keuangan (bulan ini):
- 💰 Total Penjualan: 